# Semana 20 — QLoRA: 4-bit Quantization
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_20_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-20-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-20-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_20_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install peft bitsandbytes

In [ ]:
import torch, numpy as np, json, time, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
ALVOS = ["q_proj", "k_proj", "v_proj", "o_proj"]

## Parte A — Quantization do zero (roda em CPU)

### Passo 1 — Uniform quantization
Dado um tensor `w` e `bits`, crie `2**bits` níveis igualmente espaçados entre `w.min()` e `w.max()` e troque cada peso pelo nível mais próximo.

In [ ]:
def quantizar_uniforme(w, bits=4):
    # [SEU CÓDIGO AQUI] devolva (pesos reconstruídos, índices dos níveis)

w = torch.randn(20_000) * 0.02            # pesos de uma camada: aproximadamente normais
wq, idx = quantizar_uniforme(w)
print("níveis distintos:", idx.unique().numel(), "| erro quadrático médio:", f"{((w - wq) ** 2).mean().item():.3e}")

### Passo 2 — Níveis nos quantis da normal (a ideia do NF4)
Em vez de espaçar igualmente, posicione os 16 níveis nos quantis de uma normal padrão, normalize para `[-1, 1]` e escale pelo maior valor absoluto do bloco.

In [ ]:
def quantizar_nf(w, bits=4):
    n = 2 ** bits
    # [SEU CÓDIGO AQUI] calcule os níveis pelos quantis, escale e devolva (reconstruídos, índices)

wn, idxn = quantizar_nf(w)
print(f"erro uniforme: {((w - wq) ** 2).mean().item():.3e} | erro por quantis: {((w - wn) ** 2).mean().item():.3e}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for a, i, t in zip(ax, (idx, idxn), ("uniforme: uso de cada nível", "quantis: uso de cada nível")):
    a.bar(range(16), torch.bincount(i, minlength=16).numpy()); a.set_title(t)
plt.show()

Na uniform quantization, os níveis das pontas quase não são usados. Nos quantis, o uso é equilibrado.

### Passo 3 — Bits × erro

In [ ]:
for bits in (2, 3, 4, 6, 8):
    # [SEU CÓDIGO AQUI] erro quadrático médio da quantization por quantis com `bits`
    print(f"{bits} bits | {2**bits:3d} níveis | erro {e:.3e} | memória de 7B parâmetros: {7e9 * bits / 8 / 1e9:.1f} GB")

### Passo 4 — O efeito na saída de uma layer
O que importa não é o erro no peso, e sim o erro na saída `W x`.

In [ ]:
W = torch.randn(512, 512) * 0.02; x = torch.randn(64, 512)
for nome, fn in [("uniforme 4 bits", quantizar_uniforme), ("quantis 4 bits", quantizar_nf)]:
    Wq = fn(W.flatten())[0].view_as(W)
    # [SEU CÓDIGO AQUI] erro relativo da saída: ||Wx - Wq x|| / ||Wx||
    print(f"{nome:16s} | erro relativo na saída: {erro_rel:.3%}")

## Parte B — QLoRA em um LLM (requer GPU)

### Passo 5 — Carregar a base em 4 bits

In [ ]:
if device == "cuda":
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    tok = AutoTokenizer.from_pretrained(MODELO)
    # [SEU CÓDIGO AQUI] defina bnb (BitsAndBytesConfig NF4 com double quantization) e carregue `base` com quantization_config
    n = sum(p.numel() for p in base.parameters())
    print(f"memória dos pesos em 4 bits: {base.get_memory_footprint() / 1e9:.2f} GB | em fp16 seriam ≈ {n * 2 / 1e9:.2f} GB | em fp32 ≈ {n * 4 / 1e9:.2f} GB")
else:
    print("Sem GPU: a Parte B exige a T4 do Colab (bitsandbytes usa CUDA).")

### Passo 6 — Preparar para treino e aplicar LoRA

In [ ]:
# --- Dados brutos: chamados de TI fictícios, com a sujeira típica de dados reais ---
import random
SISTEMA_TICKETS = ('Você é um classificador de chamados de TI. Responda apenas com um JSON com as chaves '
                   '"categoria", "prioridade" e "resumo".')
CATEGORIAS = ["acesso", "hardware", "software", "rede", "financeiro"]; PRIORIDADES = ["baixa", "media", "alta"]
def gerar_tickets_brutos(n=600, seed=0):
    rnd = random.Random(seed)
    modelos = {
        "acesso": [("Não consigo entrar no {x}, aparece senha inválida.", "Falha de login no {x}"),
                   ("Meu usuário foi bloqueado no {x} depois de várias tentativas.", "Usuário bloqueado no {x}")],
        "hardware": [("O {x} não liga desde ontem.", "{x} não liga"), ("A tela do {x} está piscando sem parar.", "Tela do {x} com defeito")],
        "software": [("O {x} fecha sozinho quando abro um arquivo grande.", "{x} fecha inesperadamente"),
                     ("Preciso instalar o {x} na minha máquina.", "Instalação do {x}")],
        "rede": [("A internet do {x} está muito lenta.", "Lentidão de rede no {x}"), ("O wifi do {x} cai a cada dez minutos.", "Queda de wifi no {x}")],
        "financeiro": [("Minha nota de reembolso de {x} reais não foi paga.", "Reembolso de {x} reais pendente"),
                       ("Fui cobrado em duplicidade em {x} reais na fatura.", "Cobrança duplicada de {x} reais")],
    }
    slots = {"acesso": ["SAP", "portal RH", "CRM", "e-mail corporativo", "VPN"], "hardware": ["notebook", "monitor", "teclado", "headset", "dock"],
             "software": ["Excel", "Power BI", "Teams", "VS Code", "Outlook"], "rede": ["3º andar", "escritório de Lisboa", "auditório", "sala de reuniões", "home office"],
             "financeiro": ["150", "480", "1200", "75", "320"]}
    urgencia = {"alta": ["Isso está parando toda a equipe!", "É urgente, tenho uma entrega hoje."],
                "media": ["Preciso disso resolvido esta semana.", "Está atrapalhando meu trabalho."],
                "baixa": ["Sem pressa.", "Quando puderem, por favor."]}
    saudacoes = ["Olá,", "Bom dia!", "Oi pessoal,", "Prezados,", ""]
    nomes = ["ana.souza", "joao.lima", "carla.mendes", "pedro.alves", "rita.costa"]
    linhas = []
    for _ in range(n):
        cat = rnd.choice(CATEGORIAS); texto, resumo = rnd.choice(modelos[cat]); x = rnd.choice(slots[cat]); prio = rnd.choice(PRIORIDADES)
        corpo = f"{rnd.choice(saudacoes)} {texto.format(x=x)} {rnd.choice(urgencia[prio])}"
        if rnd.random() < 0.5: corpo += f"  Att, {rnd.choice(nomes)}@empresa.com, tel (11) 9{rnd.randint(1000, 9999)}-{rnd.randint(1000, 9999)}"
        if rnd.random() < 0.2: corpo = "   " + corpo.replace(" ", "  ", 2)
        rot_cat = rnd.choice([cat, cat, cat.upper(), f" {cat} "]); rot_prio = rnd.choice([prio, prio, prio.capitalize(), "média" if prio == "media" else prio])
        linhas.append({"texto": corpo, "categoria": rot_cat, "prioridade": rot_prio, "resumo": resumo.format(x=x)})
    linhas += [dict(rnd.choice(linhas)) for _ in range(n // 12)]                      # duplicatas exatas
    linhas += [{"texto": "", "categoria": "acesso", "prioridade": "baixa", "resumo": ""} for _ in range(n // 40)]   # vazios
    rnd.shuffle(linhas); return linhas

# --- Pipeline de preparação (construído passo a passo na Semana 17) ---
import re, json, unicodedata
def preparar_tickets(n=600, seed=0):
    """Devolve (treino, val, teste): listas de {"messages": [system, user, assistant]}."""
    norm = lambda s: unicodedata.normalize("NFKD", s.strip().lower()).encode("ascii", "ignore").decode()
    vistos, exemplos = set(), []
    for l in gerar_tickets_brutos(n, seed):
        t = re.sub(r"\s+", " ", l["texto"]).strip()
        t = re.sub(r"[\w.]+@[\w.]+", "<EMAIL>", t); t = re.sub(r"\(\d{2}\)\s*9?\d{4}-\d{4}", "<TELEFONE>", t)
        if not t or t in vistos: continue
        vistos.add(t)
        alvo = {"categoria": norm(l["categoria"]), "prioridade": norm(l["prioridade"]), "resumo": l["resumo"]}
        exemplos.append({"messages": [{"role": "system", "content": SISTEMA_TICKETS}, {"role": "user", "content": t},
                                      {"role": "assistant", "content": json.dumps(alvo, ensure_ascii=False)}]})
    random.Random(seed).shuffle(exemplos); a, b = int(0.8 * len(exemplos)), int(0.9 * len(exemplos))
    return exemplos[:a], exemplos[a:b], exemplos[b:]

treino, val, teste = preparar_tickets()
# --- Utilitários de fine-tuning (escritos passo a passo na Semana 19) ---
import json, random, torch
def gerar(modelo, tok, mensagens, max_new_tokens=64):
    prompt = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    ent = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    with torch.no_grad():
        out = modelo.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=pad)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def codificar(tok, mensagens, max_len=256):
    """ids = prompt + resposta; labels = -100 no prompt (só a resposta entra na perda)."""
    prompt = tok.apply_chat_template(mensagens[:-1], tokenize=False, add_generation_prompt=True)
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(mensagens[-1]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return (ids_p + ids_r)[:max_len], ([-100] * len(ids_p) + ids_r)[:max_len]

def colar(tok, exemplos, device):
    cods = [codificar(tok, e["messages"]) for e in exemplos]; T = max(len(i) for i, _ in cods)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.tensor([i + [pad] * (T - len(i)) for i, _ in cods]); lab = torch.tensor([l + [-100] * (T - len(l)) for _, l in cods])
    mask = torch.tensor([[1] * len(i) + [0] * (T - len(i)) for i, _ in cods])
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device), "labels": lab.to(device)}

def treinar(modelo, tok, dados, epocas=2, lr=2e-4, lote=8, log=20):
    device = modelo.device; amp = device.type == "cuda"
    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    scaler = torch.amp.GradScaler(enabled=amp); hist = []; modelo.train(); rnd = random.Random(0)
    for ep in range(epocas):
        ordem = list(range(len(dados))); rnd.shuffle(ordem)
        for n, i in enumerate(range(0, len(ordem), lote)):
            b = colar(tok, [dados[j] for j in ordem[i:i + lote]], device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                loss = modelo(**b).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); hist.append(loss.item())
            if n % log == 0: print(f"época {ep} passo {n:3d} | loss {loss.item():.3f}")
    modelo.eval(); return hist

@torch.no_grad()
def perda(modelo, tok, dados, lote=8):
    modelo.eval(); device = modelo.device; tot = 0.0
    for i in range(0, len(dados), lote):
        b = colar(tok, dados[i:i + lote], device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            tot += modelo(**b).loss.item() * len(dados[i:i + lote])
    return tot / len(dados)

def avaliar_extracao(modelo, tok, dados, n=40):
    """Gera a resposta para cada exemplo e compara com o gabarito. Devolve lista de dicts."""
    linhas = []
    for e in dados[:n]:
        ouro = json.loads(e["messages"][-1]["content"]); bruto = gerar(modelo, tok, e["messages"][:-1])
        try:
            p = json.loads(bruto[bruto.index("{"): bruto.rindex("}") + 1]); ok = isinstance(p, dict)
        except Exception:
            p, ok = {}, False
        linhas.append({"json_valido": ok, "categoria_ok": p.get("categoria") == ouro["categoria"], "prioridade_ok": p.get("prioridade") == ouro["prioridade"],
                       "tudo_ok": ok and all(p.get(k) == ouro[k] for k in ouro), "cat_ouro": ouro["categoria"], "cat_pred": p.get("categoria"), "bruto": bruto})
    return linhas
if device == "cuda":
    # [SEU CÓDIGO AQUI] prepare_model_for_kbit_training + get_peft_model
    modelo.print_trainable_parameters()
    antes = pd.DataFrame(avaliar_extracao(modelo, tok, val, n=30))[["json_valido", "categoria_ok", "prioridade_ok", "tudo_ok"]].mean()
    print(antes.round(3))

### Passo 7 — Treinar e medir memória
O loop é o mesmo da Semana 19 (função `treinar` dos utilitários). Meça o pico de memória de GPU.

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    # [SEU CÓDIGO AQUI] treine por 1 epoch
    print(f"tempo: {time.time() - t0:.0f}s | pico de memória de GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    depois = pd.DataFrame(avaliar_extracao(modelo, tok, val, n=30))[["json_valido", "categoria_ok", "prioridade_ok", "tudo_ok"]].mean()
    print(pd.DataFrame({"antes": antes, "depois": depois}).round(3))
    plt.plot(hist); plt.yscale("log"); plt.title("loss de treino (QLoRA)"); plt.show()

### Passo 8 — Salvar o adapter

In [ ]:
if device == "cuda":
    modelo.save_pretrained("adaptador-qlora"); print("adaptador salvo em adaptador-qlora/")

## Teste Prático de Entrevista — Orçamento de VRAM para QLoRA (4-bit vs FP16)
Um entrevistador pede: *"Implemente `cabe_na_gpu_qlora(params_b, trainable_frac, activation_gb, gpu_vram_gb)` retornando `(vram_necessaria_gb, cabe_bool)`. Considere base quantizada em 4-bit (`0.55 bytes/param` incluindo escalas) e parâmetros LoRA treináveis consumindo `16 bytes/param` (pesos bf16 + grads + AdamW states)."* 

In [ ]:
def cabe_na_gpu_qlora(params_b, trainable_frac, activation_gb, gpu_vram_gb):
    """Estima VRAM total em GB (base 1024^3) para treino QLoRA e verifica se cabe na GPU."""
    # [SEU CÓDIGO AQUI]
    pass


vram, ok = cabe_na_gpu_qlora(params_b=7.0, trainable_frac=0.01, activation_gb=3.0, gpu_vram_gb=16.0)
assert ok is True and 7.0 < vram < 9.0, f"vram={vram}"
_, ok_70b = cabe_na_gpu_qlora(params_b=70.0, trainable_frac=0.01, activation_gb=6.0, gpu_vram_gb=24.0)
assert ok_70b is False
print(f"OK! 7B QLoRA VRAM={vram:.2f} GB (cabe em 16GB: {ok})")

## Fechamento
Compare com a Semana 19: modelo três vezes maior, memória de GPU parecida. Registre em `docs/` a tabela pedida no entregável e responda: o passo de treino ficou mais rápido ou mais lento por exemplo? Por quê?